# Carteras óptimas según la fortaleza del dólar
**Finanzas Computacionales 2026 · Trabajo Práctico Integrador**

Armamos carteras de acciones de los 11 sectores de la economía, optimizadas para **tres perfiles de riesgo** y **dos escenarios del dólar** (fuerte y débil), con un tope de **8% por acción** y **25% por sector**. Las comparamos contra la cartera óptima que **no tiene en cuenta el dólar**, con las métricas de riesgo del TP.

**Objetivo:** ver si mirar el dólar mejora la cartera, y en qué acciones y sectores conviene invertir según el perfil y el dólar.

### Cómo correrlo
1. `Entorno de ejecución → Ejecutar todas` (o `Ctrl + F9`).
2. Con `CONFIG["descargar"] = False` (lo normal) usa los datos guardados en el repositorio: no depende de Yahoo Finance y tarda poco.
3. Para actualizar los datos, poner `CONFIG["descargar"] = True` (tarda varios minutos) y seguir los pasos del `README.md`.

### Índice
0. Configuración
1. Universo de acciones
2. Datos de mercado (clase `DatosMercado`)
3. Resumen del universo
4. Señal del dólar (clase `SenalDolar`)
5. Primera mirada: retornos por sector según el dólar
6. *(próximamente)* Estimación de μ y Σ · Optimizador y carteras · Métricas de riesgo · Backtest · Stress test · Conclusiones

## 0. Configuración
Todo lo que se puede cambiar está en `CONFIG`. El resto del código **no tiene fechas ni números escritos a mano**: si cambiamos algo acá y hacemos "Ejecutar todas", todo el análisis se recalcula.

Algunos parámetros se usan en secciones que todavía no están armadas (lo indica el comentario).

In [ ]:
CONFIG = {
    # ---------------- Datos (secciones 1 y 2) ----------------
    "descargar": False,              # False: usa los datos guardados en el repositorio (presentación)
                                     # True: baja todo de Wikipedia y Yahoo Finance (varios minutos)
    "fecha_inicio": "2015-01-01",
    "fecha_fin": None,               # None = hasta hoy (solo se usa si descargar=True)
    "tolerancia_inicio_dias": 31,    # una acción entra si tiene precios desde (inicio + 31 días)
    "max_faltantes": 0.02,           # se descarta si le falta precio en más del 2% de los días
    "referencias": ["SPY", "ACWI"],  # S&P 500 y mercado mundial (para comparar)

    # ---------------- Señal del dólar (sección 4) ----------------
    "ticker_dolar": "DX-Y.NYB",      # índice del dólar (DXY) en Yahoo Finance
    "ventana_dolar": 200,            # días del promedio móvil

    # ---------------- Estimación y backtest (próximas secciones) ----------------
    "corte": "2022-12-31",           # hasta acá se estiman las carteras; después se prueban
    "rf": 0.04,                      # tasa libre de riesgo anual (Clase 8)

    # ---------------- Optimización (próximas secciones) ----------------
    "tope_accion": 0.08,             # máximo 8% por acción
    "tope_sector": 0.25,             # máximo 25% por sector
    "peso_minimo": 0.01,             # pesos menores a 1% se eliminan y se reescala
    "lambdas": {"Conservador": 20, "Moderado": 5, "Agresivo": 1},   # aversión al riesgo (a calibrar)
    "recalcular": False,             # False: usa las carteras ya calculadas y guardadas

    # ---------------- Repositorio y reproducibilidad ----------------
    "url_repo": "https://raw.githubusercontent.com/davidmarinich1-source/tp-finanzas-computacionales/main/",
    "carpeta_datos": "data",
    "semilla": 42,                   # Monte Carlo siempre da los mismos números
}

In [ ]:
import io
import os
import sys
import json
import time
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 60)
plt.rcParams["figure.dpi"] = 110

rng = np.random.default_rng(CONFIG["semilla"])   # generador aleatorio con semilla fija
print("Librerías cargadas. pandas", pd.__version__, "| numpy", np.__version__)

## 1. Universo de acciones
**Criterio (sin límite de cantidad de acciones):**
- **Todas las acciones del S&P 500.** La lista y el sector de cada una salen de Wikipedia.
- **Más ADR de empresas de otros países**, de todos los sectores. Un ADR cotiza en EE.UU. y en dólares, así que **todo queda en la misma moneda** sin hacer conversiones: el precio en dólares del ADR ya incluye cómo se movió la moneda de su país.
- Se quedan solo las acciones con precios desde enero de 2015 y sin huecos importantes (sección 2).

**Limitación a mencionar (sesgo de supervivencia):** usamos las empresas que están **hoy** en el S&P 500. Las que salieron del índice en estos años (por quiebra o mal desempeño) no aparecen, y eso puede hacer que el pasado se vea un poco mejor de lo que fue.

Los sectores usan la clasificación GICS (la misma del S&P 500), traducida al castellano.

In [ ]:
# Traducción de los 11 sectores GICS (como vienen en Wikipedia) al castellano
SECTORES_ES = {
    "Information Technology": "Tecnología",
    "Financials": "Financiero",
    "Health Care": "Salud",
    "Energy": "Energía",
    "Consumer Staples": "Consumo básico",
    "Consumer Discretionary": "Consumo discrecional",
    "Industrials": "Industria",
    "Materials": "Materiales",
    "Communication Services": "Comunicación",
    "Utilities": "Servicios públicos",
    "Real Estate": "Inmobiliario",
}

# ADR y acciones extranjeras que cotizan en EE.UU. en dólares: {sector: [(ticker, empresa, país), ...]}
# Si alguna no tiene datos suficientes, la sección 2 la descarta sola y avisa por qué.
ADRS = {
    "Tecnología": [
        ("TSM", "Taiwan Semiconductor", "Taiwán"), ("ASML", "ASML Holding", "Países Bajos"),
        ("SAP", "SAP", "Alemania"), ("INFY", "Infosys", "India"), ("WIT", "Wipro", "India"),
        ("UMC", "United Microelectronics", "Taiwán"), ("ASX", "ASE Technology", "Taiwán"),
        ("STM", "STMicroelectronics", "Suiza"), ("ERIC", "Ericsson", "Suecia"),
        ("NOK", "Nokia", "Finlandia"), ("CHKP", "Check Point Software", "Israel"),
        ("NICE", "NICE", "Israel"), ("LOGI", "Logitech", "Suiza"), ("GLOB", "Globant", "Argentina"),
    ],
    "Financiero": [
        ("HSBC", "HSBC", "Reino Unido"), ("BCS", "Barclays", "Reino Unido"), ("LYG", "Lloyds", "Reino Unido"),
        ("DB", "Deutsche Bank", "Alemania"), ("UBS", "UBS", "Suiza"), ("ING", "ING", "Países Bajos"),
        ("SAN", "Banco Santander", "España"), ("BBVA", "BBVA", "España"),
        ("MUFG", "Mitsubishi UFJ", "Japón"), ("SMFG", "Sumitomo Mitsui", "Japón"), ("MFG", "Mizuho", "Japón"),
        ("HDB", "HDFC Bank", "India"), ("IBN", "ICICI Bank", "India"),
        ("ITUB", "Itaú Unibanco", "Brasil"), ("BBD", "Bradesco", "Brasil"), ("BAP", "Credicorp", "Perú"),
        ("KB", "KB Financial", "Corea del Sur"), ("SHG", "Shinhan Financial", "Corea del Sur"),
        ("RY", "Royal Bank of Canada", "Canadá"), ("TD", "Toronto-Dominion Bank", "Canadá"),
        ("BNS", "Scotiabank", "Canadá"), ("BMO", "Bank of Montreal", "Canadá"), ("CM", "CIBC", "Canadá"),
        ("MFC", "Manulife", "Canadá"), ("SLF", "Sun Life", "Canadá"),
        ("GGAL", "Grupo Financiero Galicia", "Argentina"), ("BMA", "Banco Macro", "Argentina"),
        ("BBAR", "BBVA Argentina", "Argentina"),
    ],
    "Salud": [
        ("NVO", "Novo Nordisk", "Dinamarca"), ("AZN", "AstraZeneca", "Reino Unido"),
        ("NVS", "Novartis", "Suiza"), ("SNY", "Sanofi", "Francia"), ("GSK", "GSK", "Reino Unido"),
        ("TEVA", "Teva", "Israel"), ("PHG", "Philips", "Países Bajos"), ("SNN", "Smith & Nephew", "Reino Unido"),
    ],
    "Energía": [
        ("BP", "BP", "Reino Unido"), ("SHEL", "Shell", "Reino Unido"), ("TTE", "TotalEnergies", "Francia"),
        ("EQNR", "Equinor", "Noruega"), ("E", "Eni", "Italia"), ("PBR", "Petrobras", "Brasil"),
        ("YPF", "YPF", "Argentina"), ("TGS", "Transportadora Gas del Sur", "Argentina"),
        ("EC", "Ecopetrol", "Colombia"), ("SU", "Suncor", "Canadá"), ("CNQ", "Canadian Natural", "Canadá"),
        ("ENB", "Enbridge", "Canadá"), ("TRP", "TC Energy", "Canadá"), ("IMO", "Imperial Oil", "Canadá"),
        ("CVE", "Cenovus", "Canadá"),
    ],
    "Consumo básico": [
        ("UL", "Unilever", "Reino Unido"), ("BUD", "AB InBev", "Bélgica"), ("DEO", "Diageo", "Reino Unido"),
        ("BTI", "British American Tobacco", "Reino Unido"), ("NSRGY", "Nestlé", "Suiza"),
        ("FMX", "FEMSA", "México"), ("KOF", "Coca-Cola FEMSA", "México"), ("ABEV", "Ambev", "Brasil"),
        ("CCU", "CCU", "Chile"), ("CRESY", "Cresud", "Argentina"),
    ],
    "Consumo discrecional": [
        ("TM", "Toyota", "Japón"), ("HMC", "Honda", "Japón"), ("SONY", "Sony", "Japón"),
        ("BABA", "Alibaba", "China"), ("JD", "JD.com", "China"), ("TCOM", "Trip.com", "China"),
        ("LVMUY", "LVMH", "Francia"), ("MELI", "MercadoLibre", "Argentina"),
    ],
    "Industria": [
        ("CNI", "Canadian National Railway", "Canadá"), ("CP", "Canadian Pacific Kansas City", "Canadá"),
        ("CAE", "CAE", "Canadá"), ("TRI", "Thomson Reuters", "Canadá"), ("RELX", "RELX", "Reino Unido"),
        ("SIEGY", "Siemens", "Alemania"), ("ERJ", "Embraer", "Brasil"),
        ("PAC", "Grupo Aeroportuario del Pacífico", "México"), ("ASR", "Grupo Aeroportuario del Sureste", "México"),
        ("OMAB", "Grupo Aeroportuario Centro Norte", "México"),
    ],
    "Materiales": [
        ("BHP", "BHP", "Australia"), ("RIO", "Rio Tinto", "Reino Unido"), ("VALE", "Vale", "Brasil"),
        ("TX", "Ternium", "Argentina"), ("SQM", "SQM", "Chile"), ("CRH", "CRH", "Irlanda"),
        ("MT", "ArcelorMittal", "Luxemburgo"), ("PKX", "POSCO", "Corea del Sur"), ("GGB", "Gerdau", "Brasil"),
        ("SID", "CSN", "Brasil"), ("GFI", "Gold Fields", "Sudáfrica"), ("SCCO", "Southern Copper", "Perú"),
        ("BAK", "Braskem", "Brasil"),
    ],
    "Comunicación": [
        ("VOD", "Vodafone", "Reino Unido"), ("AMX", "América Móvil", "México"), ("BIDU", "Baidu", "China"),
        ("NTES", "NetEase", "China"), ("TEF", "Telefónica", "España"), ("CHT", "Chunghwa Telecom", "Taiwán"),
        ("TLK", "Telkom Indonesia", "Indonesia"), ("SKM", "SK Telecom", "Corea del Sur"),
        ("TU", "Telus", "Canadá"), ("BCE", "BCE", "Canadá"), ("TEO", "Telecom Argentina", "Argentina"),
    ],
    "Servicios públicos": [
        ("NGG", "National Grid", "Reino Unido"), ("KEP", "Korea Electric Power", "Corea del Sur"),
        ("SBS", "Sabesp", "Brasil"), ("CIG", "Cemig", "Brasil"), ("ELP", "Copel", "Brasil"),
        ("EBR", "Eletrobras", "Brasil"), ("PAM", "Pampa Energía", "Argentina"), ("EDN", "Edenor", "Argentina"),
        ("AQN", "Algonquin Power", "Canadá"), ("BEP", "Brookfield Renewable", "Canadá"),
    ],
    "Inmobiliario": [
        ("IRS", "IRSA", "Argentina"),
    ],
}
print("ADR cargados:", sum(len(v) for v in ADRS.values()))

## 2. Datos de mercado
La clase `DatosMercado` (Clase 4: POO) junta todo lo relacionado con los datos:

| Método | Qué hace |
|---|---|
| `construir_universo()` | Lista del S&P 500 (Wikipedia) + ADR, con sector, origen y país |
| `descargar()` | Precios ajustados por dividendos de Yahoo Finance (`auto_adjust=True`, Clase 7), en bloques de 100 |
| `filtrar()` | Saca las acciones sin datos desde 2015, que dejaron de cotizar o con muchos huecos (Clase 6) |
| `guardar()` / `cargar()` | Guarda los datos en `data/` o los lee del repositorio |
| `retornos()` | Retornos simples diarios con `pct_change()` (Clase 7) |

Todo se alinea a los días en que opera la bolsa de EE.UU. (los días con precio de SPY).

In [ ]:
class DatosMercado:
    """Arma el universo de acciones, descarga (o carga) los precios y los deja listos.

    Después de llamar a preparar() quedan disponibles:
        universo     : DataFrame (índice = ticker) con nombre, sector, origen y país
        precios      : DataFrame de precios ajustados (filas = días, columnas = tickers)
        dolar        : Serie con el índice del dólar (DXY), en su propio calendario
        referencias  : DataFrame con los precios de SPY y ACWI
        descartados  : DataFrame con las acciones que se sacaron y el motivo
    """

    URL_SP500 = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    ARCHIVOS = {
        "precios": "precios.csv.gz",
        "universo": "universo.csv",
        "dolar": "dolar.csv",
        "referencias": "referencias.csv",
        "descartados": "descartados.csv",
        "info": "info.json",
    }

    def __init__(self, config):
        self.config = config
        self.universo = None
        self.precios = None
        self.dolar = None
        self.referencias = None
        self.descartados = None
        self.info = {}

    # ------------------------------------------------------------------
    # 1) Universo
    # ------------------------------------------------------------------
    def _sp500(self):
        """Lista actual del S&P 500 desde Wikipedia."""
        # Wikipedia rechaza pedidos sin "User-Agent": por eso usamos requests con un encabezado
        html = requests.get(self.URL_SP500, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
        tabla = pd.read_html(io.StringIO(html))[0]
        return pd.DataFrame({
            "ticker": tabla["Symbol"].str.replace(".", "-", regex=False),   # BRK.B -> BRK-B (formato Yahoo)
            "nombre": tabla["Security"],
            "sector": tabla["GICS Sector"].map(SECTORES_ES),
            "origen": "EE.UU.",
            "pais": "EE.UU.",
        })

    def _adrs(self):
        """ADR de la lista ADRS (sección 1)."""
        filas = [(t, nombre, sector, "Fuera de EE.UU.", pais)
                 for sector, lista in ADRS.items() for (t, nombre, pais) in lista]
        return pd.DataFrame(filas, columns=["ticker", "nombre", "sector", "origen", "pais"])

    def construir_universo(self):
        universo = pd.concat([self._sp500(), self._adrs()], ignore_index=True)
        universo = universo.drop_duplicates(subset="ticker").set_index("ticker")
        self.universo = universo
        print(f"Universo inicial: {len(universo)} acciones "
              f"({(universo['origen'] == 'EE.UU.').sum()} del S&P 500 y "
              f"{(universo['origen'] != 'EE.UU.').sum()} ADR)")
        return self

    # ------------------------------------------------------------------
    # 2) Descarga
    # ------------------------------------------------------------------
    @staticmethod
    def _yfinance():
        """Importa yfinance; si no está instalado (raro en Colab), lo instala."""
        try:
            import yfinance as yf
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
            import yfinance as yf
        return yf

    def _bajar(self, tickers, inicio, fin, tam_bloque=100):
        """Baja precios de cierre ajustados en bloques (para no saturar a Yahoo)."""
        yf = self._yfinance()
        partes = []
        bloques = [tickers[i:i + tam_bloque] for i in range(0, len(tickers), tam_bloque)]
        for i, bloque in enumerate(bloques, start=1):
            print(f"  Bloque {i}/{len(bloques)} ({len(bloque)} tickers)...")
            datos = yf.download(bloque, start=inicio, end=fin, auto_adjust=True,
                                progress=False, threads=True)["Close"]
            if isinstance(datos, pd.Series):          # pasa cuando el bloque tiene un solo ticker
                datos = datos.to_frame(bloque[0])
            partes.append(datos)
            time.sleep(1)
        precios = pd.concat(partes, axis=1)
        precios.index = pd.to_datetime(precios.index).tz_localize(None)
        return precios

    def descargar(self):
        inicio = self.config["fecha_inicio"]
        fin = self.config["fecha_fin"]
        tickers = list(self.universo.index)
        refs = self.config["referencias"]

        print("Descargando acciones y referencias de Yahoo Finance...")
        todo = self._bajar(tickers + refs, inicio, fin)

        # Reintento único de los que vinieron vacíos (Yahoo a veces falla de a ratos)
        vacios = [t for t in tickers + refs if t not in todo.columns or todo[t].isna().all()]
        if vacios:
            print(f"Reintentando {len(vacios)} tickers sin datos...")
            otra_vez = self._bajar(vacios, inicio, fin)
            todo = todo.drop(columns=[t for t in vacios if t in todo.columns]).join(otra_vez, how="outer")

        # Calendario: días en que operó la bolsa de EE.UU. (días con precio de SPY)
        dias = todo["SPY"].dropna().index
        todo = todo.reindex(dias)
        self.referencias = todo[refs]
        self.precios = todo[[t for t in tickers if t in todo.columns]]

        # Dólar: bajamos desde un año antes para que el promedio de 200 días ya exista en enero de 2015
        inicio_dolar = (pd.Timestamp(inicio) - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
        print("Descargando el índice del dólar (DXY)...")
        dolar = self._bajar([self.config["ticker_dolar"]], inicio_dolar, fin)
        self.dolar = dolar.iloc[:, 0].dropna().rename("DXY")

        self.info = {"fecha_descarga": pd.Timestamp.today().strftime("%Y-%m-%d"),
                     "fecha_inicio": inicio, "ultimo_dato": str(self.precios.index[-1].date())}
        return self

    # ------------------------------------------------------------------
    # 3) Filtro de calidad de datos
    # ------------------------------------------------------------------
    def filtrar(self):
        precios = self.precios
        limite_inicio = pd.Timestamp(self.config["fecha_inicio"]) + pd.Timedelta(days=self.config["tolerancia_inicio_dias"])
        limite_fin = precios.index[-1] - pd.Timedelta(days=10)
        primer_dato = precios.apply(lambda s: s.first_valid_index())
        ultimo_dato = precios.apply(lambda s: s.last_valid_index())

        motivos = {}
        for t in self.universo.index:
            if t not in precios.columns or precios[t].isna().all():
                motivos[t] = "sin datos en Yahoo Finance"
            elif primer_dato[t] > limite_inicio:
                motivos[t] = f"cotiza recién desde {primer_dato[t].date()}"
            elif ultimo_dato[t] < limite_fin:
                motivos[t] = f"dejó de cotizar el {ultimo_dato[t].date()}"
            else:
                faltantes = precios.loc[primer_dato[t]:, t].isna().mean()
                if faltantes > self.config["max_faltantes"]:
                    motivos[t] = f"le falta precio en el {faltantes:.1%} de los días"

        self.descartados = (pd.DataFrame({"motivo": pd.Series(motivos, dtype="object")})
                            .join(self.universo[["nombre", "sector", "origen"]]))
        quedan = [t for t in self.universo.index if t not in motivos]
        self.universo = self.universo.loc[quedan]

        # Huecos sueltos: se repite el último precio (máximo 5 días seguidos)
        precios = precios[quedan].ffill(limit=5)
        # Arrancamos el día en que todas las acciones ya tienen precio
        precios = precios.loc[precios.notna().all(axis=1).idxmax():]
        self.precios = precios.dropna(axis=0, how="any")
        self.referencias = self.referencias.reindex(self.precios.index).ffill()

        print(f"Quedan {len(quedan)} acciones | descartadas: {len(motivos)}")
        return self

    # ------------------------------------------------------------------
    # 4) Guardar / cargar
    # ------------------------------------------------------------------
    def guardar(self):
        carpeta = self.config["carpeta_datos"]
        os.makedirs(carpeta, exist_ok=True)
        ruta = lambda clave: os.path.join(carpeta, self.ARCHIVOS[clave])
        self.precios.round(4).to_csv(ruta("precios"), compression="gzip")
        self.universo.to_csv(ruta("universo"))
        self.dolar.round(4).to_csv(ruta("dolar"))
        self.referencias.round(4).to_csv(ruta("referencias"))
        self.descartados.to_csv(ruta("descartados"))
        self.info.update({"n_acciones": int(self.precios.shape[1]),
                          "n_descartadas": int(len(self.descartados))})
        with open(ruta("info"), "w", encoding="utf-8") as f:
            json.dump(self.info, f, ensure_ascii=False, indent=2)
        print(f"Datos guardados en la carpeta '{carpeta}/'")
        return self

    def _ubicacion(self, clave):
        """Si el archivo está en la carpeta local lo usa; si no, lo lee del repositorio en GitHub."""
        local = os.path.join(self.config["carpeta_datos"], self.ARCHIVOS[clave])
        if os.path.exists(local):
            return local
        return self.config["url_repo"] + self.config["carpeta_datos"] + "/" + self.ARCHIVOS[clave]

    def cargar(self):
        leer = lambda clave, **kw: pd.read_csv(self._ubicacion(clave), index_col=0, **kw)
        print("Cargando datos guardados desde:", self._ubicacion("precios").rsplit("/", 1)[0])
        self.precios = leer("precios", parse_dates=True, compression="gzip")
        self.universo = leer("universo")
        self.dolar = leer("dolar", parse_dates=True).iloc[:, 0]
        self.referencias = leer("referencias", parse_dates=True)
        self.descartados = leer("descartados")
        ubic_info = self._ubicacion("info")
        if ubic_info.startswith("http"):
            self.info = requests.get(ubic_info, timeout=30).json()
        else:
            with open(ubic_info, encoding="utf-8") as f:
                self.info = json.load(f)
        return self

    # ------------------------------------------------------------------
    # Todo junto
    # ------------------------------------------------------------------
    def preparar(self):
        if self.config["descargar"]:
            self.construir_universo().descargar().filtrar().guardar()
        else:
            try:
                self.cargar()
            except Exception as error:
                raise RuntimeError(
                    "No se pudieron leer los datos guardados. Si todavía no se subieron al repositorio, "
                    "poner CONFIG['descargar'] = True y seguir los pasos del README.md"
                ) from error
        print(f"Listo: {self.precios.shape[1]} acciones, del {self.precios.index[0].date()} "
              f"al {self.precios.index[-1].date()} ({len(self.precios)} días)")
        return self

    def retornos(self):
        """Retornos simples diarios (Clase 7: simples para combinar activos en una cartera)."""
        return self.precios.pct_change().iloc[1:]

In [ ]:
datos = DatosMercado(CONFIG).preparar()

**Solo si descargamos datos nuevos:** la celda siguiente comprime la carpeta `data/` y la baja a la computadora, para subirla al repositorio (los pasos están en el `README.md`). Con `descargar = False` no hace nada.

In [ ]:
if CONFIG["descargar"]:
    import shutil
    shutil.make_archive("datos_tp", "zip", CONFIG["carpeta_datos"])
    try:
        from google.colab import files          # solo existe dentro de Colab
        files.download("datos_tp.zip")
        print("Se descargó datos_tp.zip: descomprimirlo y subir los archivos a la carpeta data/ del repositorio.")
    except ImportError:
        print("No estamos en Colab: los archivos quedaron en la carpeta data/ y en datos_tp.zip")
else:
    print("Usando datos guardados: no hay nada para bajar.")

## 3. Resumen del universo
Cuántas acciones quedaron por sector y origen, y cuáles se descartaron (y por qué).

In [ ]:
print("Datos descargados el:", datos.info.get("fecha_descarga", "s/d"))
conteo = pd.crosstab(datos.universo["sector"], datos.universo["origen"], margins=True, margins_name="Total")
conteo

In [ ]:
ax = (pd.crosstab(datos.universo["sector"], datos.universo["origen"])
        .sort_values("EE.UU.")
        .plot(kind="barh", stacked=True, figsize=(9, 5), color=["tab:blue", "tab:orange"]))
ax.set_title("Acciones del universo por sector y origen")
ax.set_xlabel("Cantidad de acciones")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Acciones descartadas: {len(datos.descartados)}")
datos.descartados.sort_values("motivo")

## 4. Señal del dólar
Medimos la fortaleza del dólar con el **índice DXY**, que compara el dólar contra una canasta de monedas (euro, yen, libra, etc.).

**Regla:** el dólar está **fuerte** si el DXY está por encima de su promedio móvil de 200 días, y **débil** si está por debajo. El promedio se calcula con `rolling(...).mean()` (Clase 6).

**Para no hacer trampa:** cada día usamos la señal **del día anterior** (`shift(1)`), porque al abrir el mercado todavía no sabemos cómo va a cerrar el dólar ese día.

In [ ]:
class SenalDolar:
    """Clasifica cada día como dólar 'fuerte' o 'débil'.

    Regla: fuerte si el DXY está por encima de su promedio móvil de `ventana` días.
    Se usa la señal del día anterior (shift(1)) para no usar información del futuro.
    """

    def __init__(self, dolar, ventana):
        self.dolar = dolar.dropna()
        self.ventana = ventana
        self.promedio = None
        self.regimen = None

    def calcular(self):
        self.promedio = self.dolar.rolling(self.ventana).mean()
        fuerte = self.dolar > self.promedio
        regimen = pd.Series(np.where(fuerte, "fuerte", "débil"), index=self.dolar.index)
        regimen = regimen[self.promedio.notna()]      # los primeros días todavía no tienen promedio
        self.regimen = regimen.shift(1).dropna()      # señal de ayer
        return self

    def alinear(self, fechas):
        """Lleva la señal a las fechas de las acciones (el DXY tiene otro calendario).
        Para cada fecha toma la última señal conocida."""
        return self.regimen.reindex(fechas, method="ffill").dropna()

    def resumen(self, fechas, corte):
        """Días de cada escenario, separando el período de estimación y el de prueba."""
        reg = self.alinear(fechas)
        periodo = np.where(reg.index <= pd.Timestamp(corte), "Estimación (hasta el corte)", "Prueba (después del corte)")
        tabla = pd.crosstab(reg, periodo, margins=True, margins_name="Total")
        tabla.index.name = "Dólar"
        tabla.columns.name = "Período"
        return tabla

    def cambios_por_anio(self, fechas):
        """Cuántas veces por año cambia de escenario (para saber si la señal es estable)."""
        reg = self.alinear(fechas)
        cambios = (reg != reg.shift(1)).iloc[1:]
        return cambios.groupby(cambios.index.year).sum()

    def graficar(self, desde):
        dxy = self.dolar.loc[desde:]
        prom = self.promedio.loc[desde:]
        fuerte = (self.regimen.loc[desde:] == "fuerte")
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(dxy.index, dxy, lw=1, color="black", label="DXY")
        ax.plot(prom.index, prom, lw=1.5, color="tab:red", label=f"Promedio {self.ventana} días")
        ax.fill_between(fuerte.index, 0, 1, where=fuerte.values, transform=ax.get_xaxis_transform(),
                        color="tab:green", alpha=0.15, label="Dólar fuerte")
        ax.set_title("Índice del dólar (DXY) y escenarios")
        ax.legend(loc="upper left")
        plt.tight_layout()
        plt.show()

In [ ]:
senal = SenalDolar(datos.dolar, CONFIG["ventana_dolar"]).calcular()
senal.graficar(desde=CONFIG["fecha_inicio"])
senal.resumen(datos.precios.index, CONFIG["corte"])

**Cuántas veces por año cambia el escenario.** Si cambiara todos los días, la señal sería ruido y la estrategia dinámica rotaría demasiado.

In [ ]:
senal.cambios_por_anio(datos.precios.index).to_frame("cambios de escenario")

## 5. Primera mirada: retornos por sector según el dólar
Antes de optimizar, miramos si el dólar **cambia algo**: retorno promedio anual de las acciones de cada sector en días de dólar fuerte y débil.

- Solo usamos el **período de estimación** (hasta el corte), para no mirar los datos que después vamos a usar de prueba.
- Retorno anual = promedio de retornos diarios × 252 (Clase 7). Dentro de cada sector, todas las acciones pesan igual.
- Si la columna `diferencia` es grande, el dólar importa para ese sector.

In [ ]:
retornos = datos.retornos()
reg = senal.alinear(retornos.index)
ret_est = retornos.loc[reg.index].loc[:CONFIG["corte"]]     # período de estimación con señal
reg_est = reg.loc[ret_est.index]

# Retorno medio anual de cada acción en cada escenario (filas: escenario, columnas: acciones)
media_por_escenario = ret_est.groupby(reg_est).mean() * 252

# Promedio por sector y por origen
por_sector = media_por_escenario.T.groupby(datos.universo["sector"]).mean()
por_sector["diferencia"] = por_sector["fuerte"] - por_sector["débil"]
por_origen = media_por_escenario.T.groupby(datos.universo["origen"]).mean()
por_origen["diferencia"] = por_origen["fuerte"] - por_origen["débil"]

print("Retorno medio anual por origen")
display(por_origen[["fuerte", "débil", "diferencia"]].style.format("{:.1%}"))
print("Retorno medio anual por sector")
display(por_sector[["fuerte", "débil", "diferencia"]].sort_values("diferencia").style.format("{:.1%}"))

In [ ]:
ax = por_sector[["fuerte", "débil"]].sort_values("fuerte").plot(
    kind="barh", figsize=(9, 5), color=["tab:green", "tab:gray"])
ax.set_title(f"Retorno medio anual por sector según el dólar (hasta {CONFIG['corte'][:4]})")
ax.set_xlabel("Retorno anual promedio")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.axvline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

## 6. Próximas secciones (en construcción)
- **6. Estimación de μ y Σ** por escenario (todos los días, dólar fuerte, dólar débil), vectorizada y con Ledoit-Wolf.
- **7. Optimizador y carteras**: 3 perfiles × 3 escenarios + 3 estrategias dinámicas.
- **8. Métricas de riesgo**: volatilidad, VaR (paramétrico, histórico y Monte Carlo), CVaR, drawdown máximo, volatilidad rolling, Sharpe, alfa y beta (CAPM), tracking error, information ratio.
- **9. Backtest** fuera de muestra (después del corte).
- **10. Stress test**.
- **11. Conclusiones**.